# Interacting with Deployed Agents on Vertex AI Agent Engine

This notebook shows how to connect to, query, and manage agents deployed to [Vertex AI Agent Engine](https://docs.cloud.google.com/agent-builder/agent-engine/overview) using both the **Python SDK** and **REST API**.

**What you'll learn:**
- Connect to a deployed agent
- Create and manage sessions
- Send queries and stream responses
- List and clean up sessions
- Use the REST API directly

**Prerequisites:**
- An agent deployed via `deploy.py` (see [deploy/readme.md](readme.md))
- `google-cloud-aiplatform >= 1.112.0` installed
- Authenticated with `gcloud auth application-default login`

---
## Setup

In [1]:
PROJECT_ID = "statmike-mlops-349915"
LOCATION = "us-central1"

In [2]:
import vertexai
import json

vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

---
## Find Your Deployed Agents

List all Agent Engine deployments in the project, or filter by name:

In [3]:
# List all deployed agents
for agent in client.agent_engines.list():
    print(f"{agent.api_resource.display_name:40s} {agent.api_resource.name}")

data-onboarding-chat                     projects/1026793852137/locations/us-central1/reasoningEngines/4403271940115005440
agent_convo_api                          projects/1026793852137/locations/us-central1/reasoningEngines/6919838002059411456
document_agent                           projects/1026793852137/locations/us-central1/reasoningEngines/1988805428414251008
agent_bq_forecast                        projects/1026793852137/locations/us-central1/reasoningEngines/2255516162987130880


In [4]:
# Filter by display name
for agent in client.agent_engines.list(config={"filter": 'display_name="data-onboarding-chat"'}):
    print(f"{agent.api_resource.display_name:40s} {agent.api_resource.name}")

data-onboarding-chat                     projects/1026793852137/locations/us-central1/reasoningEngines/4403271940115005440


---
## Connect to a Deployed Agent

Use the resource name from `deploy.py --info` or from the list above:

In [5]:
# Load the resource name from deployment.json
with open("chat/deployment.json") as f:
    RESOURCE_NAME = json.load(f)["resource_name"]

print(RESOURCE_NAME)

projects/1026793852137/locations/us-central1/reasoningEngines/4403271940115005440


In [6]:
# Get the deployed agent
agent = client.agent_engines.get(name=RESOURCE_NAME)
print(f"Connected to: {agent.api_resource.display_name}")

Connected to: data-onboarding-chat


---
## Sessions

Agent Engine provides persistent, cloud-backed sessions. Each session maintains conversation history for a user.

### Create a Session

In [7]:
USER_ID = "notebook-user"

session = await agent.async_create_session(user_id=USER_ID)
SESSION_ID = session["id"]

print(f"Session ID: {SESSION_ID}")

Session ID: 2079387182040088576


### Send a Query

Queries stream back events — each event can contain agent transfers, tool calls, or text responses. The helper below extracts just the final text:

In [8]:
def extract_text(event: dict) -> str | None:
    """Pull text from a streaming event, if present."""
    content = event.get("content")
    if not content:
        return None
    parts = content.get("parts", [])
    texts = [p["text"] for p in parts if "text" in p]
    return "\n".join(texts) if texts else None

In [9]:
async for event in agent.async_stream_query(
    user_id=USER_ID,
    session_id=SESSION_ID,
    message="What tables are available?",
):
    text = extract_text(event)
    if text:
        print(text)

I found 58 tables. Here's a summary of the tables and some of their columns:

*   **bats_listed_short_interest_finra** (`1026793852137.data_onboarding_datashop_cboe_com_bronze.bats_listed_short_interest_finra`)
    *   Columns: `cycle_settlement_date`, `bats_symbol`, `security_name`, `shares_net_short_current_cycle`, `shares_net_short_previous_cycle`, `cycle_avg_daily_trade_volume`, `days_to_cover`, `split_indicator`, `manual_revision_indicator`, `percent_change_in_short_position`, `change_in_short_position`

*   **bbo_2020_01_02_cadjpy** (`1026793852137.data_onboarding_datashop_cboe_com_bronze.bbo_2020_01_02_cadjpy`)
    *   Columns: `trade_timestamp`, `aggressor_side`, `price`, `size`, `price_1`

*   **bbo_ldn_2023_08_22_cadjpy** (`1026793852137.data_onboarding_datashop_cboe_com_bronze.bbo_ldn_2023_08_22_cadjpy`)
    *   Columns: `trade_timestamp`, `aggressor_side`, `price`, `size`, `price_1`

*   **bbo_ny_2020_01_02_cadjpy** (`1026793852137.data_onboarding_datashop_cboe_com_bronze.b

### Follow-Up Questions

Same session = conversation history is preserved. The agent remembers context from the previous question:

In [10]:
async for event in agent.async_stream_query(
    user_id=USER_ID,
    session_id=SESSION_ID,
    message="Which of those have the most columns?",
):
    text = extract_text(event)
    if text:
        print(text)

The table with the most columns is `high_level_option_sentiment_complete` with 80 columns.

Other tables with a high number of columns include:
*   `volatility_surfaces_constant_maturity_price_relative` (55 columns)
*   `volatility_surfaces_expiration_specific_price_relative` (55 columns)
*   `volatility_surfaces_constant_maturity_delta_relative` (49 columns)
*   `volatility_surfaces_expiration_specific_delta_relative` (49 columns)
*   `rts13_public_trade_data_apa` (38 columns)


### Inspect Raw Events

To see the full event stream (transfers, tool calls, function responses), iterate without filtering:

In [11]:
# Create a fresh session for this example
session2 = await agent.async_create_session(user_id=USER_ID)

async for event in agent.async_stream_query(
    user_id=USER_ID,
    session_id=session2["id"],
    message="What does the column PRVDR_NUM mean?",
):
    content = event.get("content", {})
    parts = content.get("parts", [])
    for part in parts:
        if "function_call" in part:
            call = part["function_call"]
            print(f"  TOOL CALL: {call['name']}({json.dumps(call.get('args', {}), indent=2)})")
        elif "function_response" in part:
            resp = part["function_response"]
            # Truncate long responses for readability
            result = json.dumps(resp.get("response", {}))
            print(f"  TOOL RESULT: {resp['name']} → {result[:200]}...")
        elif "text" in part:
            print(f"  TEXT: {part['text'][:300]}")

  TOOL CALL: transfer_to_agent({
  "agent_name": "agent_catalog"
})
  TOOL RESULT: transfer_to_agent → {"result": null}...
  TOOL CALL: search_context({
  "query": "What does PRVDR_NUM mean?"
})
  TOOL RESULT: search_context → {"result": "Found 10 relevant chunk(s):\n\n---\n**[column_description]** data_onboarding_datashop_cboe_com_bronze.ta13_f_comp\nDistance: 0.7539\nColumn: data_onboarding_datashop_cboe_com_bronze.ta13_f...
  TEXT: I couldn't find any specific documentation for the column `PRVDR_NUM`. The search returned descriptions of other columns that were semantically similar but not a direct match.


---
## Session Management

### List All Sessions for a User

In [12]:
sessions = await agent.async_list_sessions(user_id=USER_ID)

for s in sessions["sessions"]:
    print(f"  {s['id']}")

  1773142407378894848
  2079387182040088576


### Get a Session

Retrieve a specific session to inspect its state:

In [13]:
session_info = await agent.async_get_session(user_id=USER_ID, session_id=SESSION_ID)

print(f"Session ID:  {session_info['id']}")
print(f"User ID:     {session_info['userId']}")
print(f"Events:      {len(session_info.get('events', []))} events in history")

Session ID:  2079387182040088576
User ID:     notebook-user
Events:      8 events in history


### Delete a Session

In [14]:
# Delete the second session we created
await agent.async_delete_session(user_id=USER_ID, session_id=session2["id"])
print(f"Deleted session: {session2['id']}")

Deleted session: 1773142407378894848


### Delete All Sessions for a User

In [15]:
sessions = await agent.async_list_sessions(user_id=USER_ID)
for s in sessions["sessions"]:
    await agent.async_delete_session(user_id=USER_ID, session_id=s["id"])
    print(f"  Deleted: {s['id']}")

print(f"\nCleaned up {len(sessions['sessions'])} session(s)")

  Deleted: 2079387182040088576

Cleaned up 1 session(s)


---
## REST API

You can also interact with Agent Engine using the REST API directly. This is useful for integrations from non-Python environments or for debugging.

The base URL pattern is:
```
https://{LOCATION}-aiplatform.googleapis.com/v1/{RESOURCE_NAME}
```

In [16]:
import google.auth
import google.auth.transport.requests
import requests

def get_access_token() -> str:
    """Get an OAuth2 access token from application default credentials."""
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    return credentials.token

BASE_URL = f"https://{LOCATION}-aiplatform.googleapis.com/v1/{RESOURCE_NAME}"
HEADERS = {"Content-Type": "application/json"}

### Create a Session (REST)

In [17]:
response = requests.post(
    f"{BASE_URL}:query",
    headers={**HEADERS, "Authorization": f"Bearer {get_access_token()}"},
    json={
        "class_method": "async_create_session",
        "input": {"user_id": "rest-user"},
    },
)

session_data = response.json()
REST_SESSION_ID = session_data["output"]["id"]
print(f"Session ID: {REST_SESSION_ID}")

Session ID: 5190248624646258688


### Stream a Query (REST)

The `streamQuery` endpoint returns Server-Sent Events (SSE). Each `data:` line is a JSON event:

In [18]:
response = requests.post(
    f"{BASE_URL}:streamQuery?alt=sse",
    headers={**HEADERS, "Authorization": f"Bearer {get_access_token()}"},
    json={
        "class_method": "async_stream_query",
        "input": {
            "user_id": "rest-user",
            "session_id": REST_SESSION_ID,
            "message": "What tables are available?",
        },
    },
    stream=True,
)

# Parse the SSE stream
for line in response.iter_lines():
    line = line.decode("utf-8")
    if line.startswith("data: "):
        event = json.loads(line[6:])
        text = extract_text(event)
        if text:
            print(text)

### Delete a Session (REST)

In [19]:
response = requests.post(
    f"{BASE_URL}:query",
    headers={**HEADERS, "Authorization": f"Bearer {get_access_token()}"},
    json={
        "class_method": "async_delete_session",
        "input": {"user_id": "rest-user", "session_id": REST_SESSION_ID},
    },
)

print(f"Deleted session: {REST_SESSION_ID} (status: {response.status_code})")

Deleted session: 5190248624646258688 (status: 200)


---
## curl Equivalents

For quick testing from the terminal:

```bash
# Set variables
export PROJECT_ID="your-project"
export LOCATION="us-central1"
export RESOURCE_ID="your-resource-id"
export BASE="https://${LOCATION}-aiplatform.googleapis.com/v1/projects/${PROJECT_ID}/locations/${LOCATION}/reasoningEngines/${RESOURCE_ID}"
export TOKEN=$(gcloud auth print-access-token)

# Create session
curl -s -X POST "${BASE}:query" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{"class_method": "async_create_session", "input": {"user_id": "curl-user"}}'

# Stream a query
curl -s -X POST "${BASE}:streamQuery?alt=sse" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{
    "class_method": "async_stream_query",
    "input": {
      "user_id": "curl-user",
      "session_id": "SESSION_ID",
      "message": "What tables are available?"
    }
  }'
```

---
## References

- [Vertex AI Agent Engine overview](https://docs.cloud.google.com/agent-builder/agent-engine/overview)
- [Test deployed agents](https://google.github.io/adk-docs/deploy/agent-engine/deploy/#test-deployed-agents)
- [Manage deployed agents](https://docs.cloud.google.com/agent-builder/agent-engine/manage)
- [SDK migration guide (v1.112.0+)](https://docs.cloud.google.com/agent-builder/deprecations/agent-engine-migration)